In [2]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

cal_masks_path = '/home/xkw/pxlames/segmentation/outputs/testResults/FIVE_BceDice_0.0001_4/cal_gt_masks.npy'
cal_scores_path = '/home/xkw/pxlames/segmentation/outputs/testResults/FIVE_BceDice_0.0001_4/cal_scores.npy'
test_masks_path = '/home/xkw/pxlames/segmentation/outputs/testResults/FIVE_BceDice_0.0001_4/gt_masks.npy'
test_scores_path = '/home/xkw/pxlames/segmentation/outputs/testResults/FIVE_BceDice_0.0001_4/scores.npy'
cal_masks = np.load(cal_masks_path)
cal_scores = np.load(cal_scores_path)
test_masks = np.load(test_masks_path)
test_scores = np.load(test_scores_path)

# 重塑数组
cal_masks = np.transpose(cal_masks, (1, 2, 0))
cal_scores = np.transpose(cal_scores, (1, 2, 0))
test_masks = np.transpose(test_masks, (1, 2, 0))
test_scores = np.transpose(test_scores, (1, 2, 0))

print(cal_scores.shape)
print("成功加载gt_masks和scores文件")


# 定义切片参数
patch_size = 2048  # 每个切片的大小
stride = 2048  # 切片的步长

# 创建切片
cal_patches_scores = []
cal_patches_masks = []
test_patches_scores = []
test_patches_masks = []

h, w = cal_scores.shape[0], cal_scores.shape[1]
for img_idx in range(cal_scores.shape[2]):
    for i in range(0, h-patch_size+1, stride):
        for j in range(0, w-patch_size+1, stride):
            cal_patches_scores_patch = cal_scores[i:i+patch_size, j:j+patch_size, img_idx]
            cal_patches_masks_patch= cal_masks[i:i+patch_size, j:j+patch_size, img_idx]
            if np.any(cal_patches_masks_patch):
                cal_patches_scores.append(cal_patches_scores_patch)
                cal_patches_masks.append(cal_patches_masks_patch)
print("校准集切片完成")

for img_idx in range(test_scores.shape[2]):
    for i in range(0, h-patch_size+1, stride):
        for j in range(0, w-patch_size+1, stride):
            test_patches_scores_patch = test_scores[i:i+patch_size, j:j+patch_size, img_idx]
            test_patches_masks_patch = test_masks[i:i+patch_size, j:j+patch_size, img_idx]
            if np.any(test_patches_masks_patch):
                test_patches_scores.append(test_patches_scores_patch)
                test_patches_masks.append(test_patches_masks_patch)
print("测试集切片完成")
# 将切片转换为numpy数组
cal_patches_scores = np.array(cal_patches_scores)
cal_patches_masks = np.array(cal_patches_masks)
test_patches_scores = np.array(test_patches_scores)
test_patches_masks = np.array(test_patches_masks)

# 重塑数组为所需的形状
cal_patches_scores = np.transpose(cal_patches_scores, (1, 2, 0))
cal_patches_masks = np.transpose(cal_patches_masks, (1, 2, 0))
test_patches_scores = np.transpose(test_patches_scores, (1, 2, 0))
test_patches_masks = np.transpose(test_patches_masks, (1, 2, 0))

print(f"切片后的形状:")
print(f"patches_scores形状: {cal_patches_scores.shape}")
print(f"patches_masks形状: {cal_patches_masks.shape}")


print(f"\n数据集划分:")
print(f"校准集大小: {cal_patches_scores.shape[2]} 个切片")
print(f"验证集大小: {test_patches_scores.shape[2]} 个切片")

(2048, 2048, 240)
成功加载gt_masks和scores文件
校准集切片完成
测试集切片完成
切片后的形状:
patches_scores形状: (2048, 2048, 240)
patches_masks形状: (2048, 2048, 240)

数据集划分:
校准集大小: 240 个切片
验证集大小: 30 个切片


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import io

# # 文件路径
# gt_masks_path = '/home/pxl/myProject/血管分割/molong-深度插值/code/gt_masks.npy'
# scores_path = '/home/pxl/myProject/血管分割/molong-深度插值/code/scores.npy'

# # 读取保存的 numpy 数组文件
# gt_masks = np.load(gt_masks_path)  # 加载真实标签数据
# scores = np.load(scores_path)      # 加载预测结果数据

# # 修改 gt_masks 和 scores 的形状，将最后一个维度转到第三个维度
# gt_masks = np.transpose(gt_masks, (1, 2, 0))
# scores = np.transpose(scores, (1, 2, 0))
# print("gt_masks 和 scores 文件已成功加载。")
# print(f"gt_masks 的形状: {gt_masks.shape}")
# print(f"scores 的形状: {scores.shape}")

# cal_idx = np.arange(0, 15)
# print(cal_idx)
# val_idx = np.setdiff1d(np.arange(1, scores.shape[2]), cal_idx) # 最后剩下的用于校准集
# print(val_idx)

# # 提取校准集的 score 数据
# cal_scores = scores[:, :, cal_idx]  # 索引从 0 开始，cal_idx 减去 1
# val_scores = scores[:, :, val_idx]  # 索引从 0 开始，cal_idx 减去 1
# cal_gt_masks = gt_masks[:, :, cal_idx]  # 索引从 0 开始，cal_idx 减去 1
# val_gt_masks = gt_masks[:, :, val_idx]  # 索引从 0 开始，cal_idx 减去 1


def create_segmentation_gif(image, initial_threshold=0.9, final_threshold=0.1, steps=20, save_path='segmentation_process.gif'):
    # 生成一系列阈值，逐步降低
    thresholds = np.linspace(initial_threshold, final_threshold, steps)
    
    # 用于存储所有帧的列表
    frames = []

    # 生成每个阈值的分割图像并保存到 frames 中
    for threshold in thresholds:
        # 生成当前阈值下的分割结果
        mask = image > threshold

        # 创建带透明度的颜色分割叠加图
        fig, ax = plt.subplots(figsize=(6, 6))
        # ax.imshow(image, cmap='gray', alpha=0.6)  # 原始图像
        ax.imshow(mask)  # 分割结果叠加

        # 设置标题和去除坐标轴
        ax.set_title(f'Threshold: {threshold:.2f}')
        ax.axis('off')
        
        # 将当前图保存为图像格式
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        frame = Image.open(buf)
        frames.append(frame)
        plt.close(fig)
    
    # 保存所有帧为 GIF
    frames[0].save(save_path, format='GIF', append_images=frames[1:], save_all=True, duration=200, loop=0)
    print(f'GIF 已保存至 {save_path}')

# 示例调用
# 假设 val_scores[:, :, no] 是分割的原始概率图像
# no = 2
id = np.random.randint(0, cal_patches_scores.shape[2])
#id = 24193
scores = cal_patches_scores[:,:,id]
mask = cal_patches_masks[:,:,id]
create_segmentation_gif(scores, initial_threshold=1, final_threshold=0, steps=100, save_path='./FIVE.gif')

GIF 已保存至 ./FIVE.gif
